# Deterministic Calendar Scheduling with LangGraph + MCP

LLMs score below 50% on temporal reasoning tasks ([OOLONG benchmark](https://arxiv.org/abs/2511.02817)). Earlier models scored as low as 29% on scheduling and 13% on duration calculations ([Test of Time, ICLR 2025](https://arxiv.org/abs/2406.09170)). Ask "Schedule for next Tuesday at 2pm" and the model picks the wrong Tuesday. Ask "Am I free at 3pm?" and it checks the wrong timezone. Then it double-books your calendar.

This notebook shows how to build a scheduling agent that **never hallucinates dates** and **never double-books** — by connecting LangGraph to a calendar MCP server ([Temporal Cortex](https://github.com/temporal-cortex/mcp)) via `langchain-mcp-adapters`. The MCP server provides deterministic datetime resolution, cross-provider availability merging, and atomic booking with Two-Phase Commit.

## Setup

You'll need:
- An **Anthropic API key** ([console.anthropic.com](https://console.anthropic.com/))
- **Node.js 18+** (for `npx` to run the MCP server locally)
- **At least one calendar provider** authenticated — run `npx @temporal-cortex/cortex-mcp auth google` first

Or for **Platform Mode** (no Node.js needed):
- A **Temporal Cortex API key** ([app.temporal-cortex.com](https://app.temporal-cortex.com)) with at least one calendar connected

In [ ]:
%pip install langchain-mcp-adapters langgraph langchain-anthropic python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set your API key (or use a .env file)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# For Local Mode (stdio), set calendar credentials:
# os.environ["GOOGLE_CLIENT_ID"] = "your-id.apps.googleusercontent.com"
# os.environ["GOOGLE_CLIENT_SECRET"] = "your-secret"

# For Platform Mode (HTTP), set your API key:
# os.environ["TEMPORAL_CORTEX_API_KEY"] = "sk_live_..."

## Step 1: Connect to the Calendar MCP Server

`MultiServerMCPClient` from `langchain-mcp-adapters` launches the MCP server, discovers all available tools, and converts them into LangChain-compatible `StructuredTool` objects. This works with any MCP server — Temporal Cortex exposes up to 15 tools across 5 layers.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Local Mode (stdio) — runs the MCP server via npx
client = MultiServerMCPClient(
    {
        "temporal-cortex": {
            "command": "npx",
            "args": ["-y", "@temporal-cortex/cortex-mcp"],
            "env": {
                "GOOGLE_CLIENT_ID": os.getenv("GOOGLE_CLIENT_ID", ""),
                "GOOGLE_CLIENT_SECRET": os.getenv("GOOGLE_CLIENT_SECRET", ""),
                "TIMEZONE": os.getenv("TIMEZONE", ""),
            },
            "transport": "stdio",
        }
    }
)

# For Platform Mode, replace the config above with:
# client = MultiServerMCPClient(
#     {
#         "temporal-cortex": {
#             "url": "https://mcp.temporal-cortex.com/mcp",
#             "headers": {"Authorization": f"Bearer {os.getenv('TEMPORAL_CORTEX_API_KEY', '')}"},
#             "transport": "streamable_http",
#         }
#     }
# )

await client.__aenter__()
tools = client.get_tools()
print(f"Discovered {len(tools)} tools: {[t.name for t in tools]}")

## Step 2: Build a ReAct Scheduling Agent

The system prompt encodes a deterministic workflow: orient in time → resolve datetime → discover calendars → find availability → book. This prevents the model from guessing dates or skipping availability checks.

We start with a **read-only query** (availability check) so the agent doesn't modify your calendar.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent

model = ChatAnthropic(model="claude-sonnet-4-6")

SYSTEM_PROMPT = (
    "You schedule meetings using Temporal Cortex calendar tools.\n\n"
    "Follow this workflow:\n"
    "1. Call get_temporal_context to learn the current time and timezone\n"
    "2. Call resolve_datetime to convert human expressions to RFC 3339 timestamps\n"
    "3. Call list_calendars to discover connected calendars\n"
    "4. Call find_free_slots to check availability on the target date\n"
    "5. Call book_slot to book the meeting (Two-Phase Commit prevents double-bookings)\n\n"
    "When calling data tools (list_calendars, list_events, find_free_slots, "
    "expand_rrule, get_availability), pass format='json' for structured output.\n"
    "Always use provider-prefixed calendar IDs (e.g., google/primary).\n"
    "Never guess dates or times — always use the tools."
)

agent = create_react_agent(model, tools, prompt=SYSTEM_PROMPT)
print(f"ReAct agent ready with {len(tools)} tools.")

In [ ]:
result = await agent.ainvoke(
    {"messages": [("user", "Am I free next Tuesday at 2pm? Check my calendar availability.")]}
)

# Print the final assistant message
for msg in reversed(result["messages"]):
    if msg.type == "ai" and msg.content:
        print(msg.content)
        break

## Step 3: Multi-Agent StateGraph

For production workflows, use a **StateGraph** with specialized nodes. Each node is a scoped ReAct sub-agent with access to only its layer's tools. The graph enforces the correct tool-calling order through explicit edges.

This is LangGraph's core differentiator: **deterministic routing**. The LLM decides what to say within each node, but the graph decides which node runs next.

In [ ]:
from typing import Annotated
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict


class SchedulingState(TypedDict):
    messages: Annotated[list, add_messages]
    temporal_context: str
    available_slots: str


async def temporal_analyst_node(state: SchedulingState) -> dict:
    """Orient in time and resolve datetime expressions (Layer 1)."""
    temporal_tools = [t for t in tools if t.name in {
        "get_temporal_context", "resolve_datetime",
        "convert_timezone", "compute_duration", "adjust_timestamp",
    }]
    sub_agent = create_react_agent(
        model, temporal_tools,
        prompt=(
            "Call get_temporal_context then resolve_datetime. "
            "Pass format='json' to data tools. Return a summary."
        ),
    )
    result = await sub_agent.ainvoke({"messages": state["messages"]})
    context = result["messages"][-1].content
    return {
        "temporal_context": context,
        "messages": [AIMessage(content=f"[Temporal Analyst] {context}")],
    }


async def calendar_manager_node(state: SchedulingState) -> dict:
    """Discover calendars and find available slots (Layer 2)."""
    cal_tools = [t for t in tools if t.name in {
        "list_calendars", "list_events", "find_free_slots",
        "expand_rrule", "check_availability",
    }]
    sub_agent = create_react_agent(
        model, cal_tools,
        prompt=(
            "Call list_calendars then find_free_slots. "
            "Use provider-prefixed IDs. Pass format='json'. Return available slots."
        ),
    )
    messages = state["messages"] + [
        HumanMessage(content=f"Temporal context: {state['temporal_context']}")
    ]
    result = await sub_agent.ainvoke({"messages": messages})
    slots = result["messages"][-1].content
    return {
        "available_slots": slots,
        "messages": [AIMessage(content=f"[Calendar Manager] {slots}")],
    }


# Build the graph
graph = StateGraph(SchedulingState)
graph.add_node("temporal_analyst", temporal_analyst_node)
graph.add_node("calendar_manager", calendar_manager_node)
graph.set_entry_point("temporal_analyst")
graph.add_edge("temporal_analyst", "calendar_manager")
graph.add_edge("calendar_manager", END)

scheduling_graph = graph.compile()
print("StateGraph compiled: temporal_analyst → calendar_manager → END")

In [ ]:
result = await scheduling_graph.ainvoke({
    "messages": [HumanMessage(content="Am I free next Tuesday at 2pm?")],
    "temporal_context": "",
    "available_slots": "",
})

print("Temporal context:", result["temporal_context"][:200], "...")
print()
print("Available slots:", result["available_slots"][:200], "...")

## What Happened Under the Hood

When the agent ran, it followed the deterministic workflow encoded in its system prompt:

1. **`get_temporal_context`** — The agent learned the current time, timezone, UTC offset, and DST status. This prevents timezone mistakes (e.g., scheduling in UTC when the user is in EST).

2. **`resolve_datetime`** — The expression "next Tuesday at 2pm" was resolved to a precise RFC 3339 timestamp. The MCP server uses deterministic date math — no LLM guessing which Tuesday is "next."

3. **`list_calendars` → `find_free_slots`** — The agent discovered connected calendar providers and checked actual availability, rather than assuming the time slot was open.

For the StateGraph version, each step ran in a **dedicated node** with access to only its layer's tools. The graph's explicit edges guaranteed the correct order: temporal analysis → calendar query. No LLM routing decisions were needed.

The key insight: the agent **never guessed** any temporal value. Every date, time, and timezone came from deterministic tools. This is what brings scheduling accuracy from ~50% (LLM inference) to 100% (tool-assisted).

In [ ]:
# Cleanup: close the MCP client
await client.__aexit__(None, None, None)
print("MCP client closed.")

## Next Steps

- **Booking with approval gates**: Use LangGraph's `interrupt_before` to pause before `book_slot`. See [human_in_the_loop.py](https://github.com/temporal-cortex/mcp/blob/main/examples/langgraph/human_in_the_loop.py).
- **Full multi-agent booking**: Add a `booking_coordinator` node with Two-Phase Commit. See [multi_agent.py](https://github.com/temporal-cortex/mcp/blob/main/examples/langgraph/multi_agent.py).
- **Tool reference**: Full input/output schemas for all 15 tools at [temporal-cortex.com/docs](https://temporal-cortex.com/docs/tool-reference).
- **Agent Skills**: Procedural knowledge for calendar workflows at [github.com/temporal-cortex/skills](https://github.com/temporal-cortex/skills).
- **LangGraph docs**: Official reference at [langchain-ai.github.io/langgraph](https://langchain-ai.github.io/langgraph/).